# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import pandas as pd

# Load dataset — single source of truth for all cells below
dataset = pd.read_csv('work/outputs/dataset.csv')

# Define once — used in every subsequent cell
feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]

leak_cols = [
    'impressions_mar', 'impressions_apr', 'pct_change',
    'trend_direction', 'trend_pct'
]

print("=== UNIT OF ANALYSIS ===")
print(f"One row = one content item per client over a 90-day feature window")
print(f"Feature window:  2026-01-01 → 2026-03-31")
print(f"Label window:    2026-04-01 → 2026-04-30")
print(f"\nTotal items:     {len(dataset):,}")
print(f"Total clients:   {dataset['client_hash_id'].nunique()}")
print(f"Total columns:   {dataset.shape[1]}")

=== UNIT OF ANALYSIS ===
One row = one content item per client over a 90-day feature window
Feature window:  2026-01-01 → 2026-03-31
Label window:    2026-04-01 → 2026-04-30

Total items:     103,691
Total clients:   42
Total columns:   17


## 1. Unit of analysis + time window

One row = one content item (`content_hash_id`) belonging to one client (`client_hash_id`),
evaluated over a fixed 90-day feature window followed by a 30-day label window.

Source: `fact_content_daily_performance` warehouse table (FlyRank ML Internship dataset,
build v20260703), filtered to `gsc_data_available = TRUE`

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Label
| Column | Description |
|---|---|
| `is_declining_label` | 1 if GSC impressions dropped ≥30% from March to April 2026 |

### Features
| Column | Description |
|---|---|
| `impressions_90d` | Total GSC impressions over 90-day feature window |
| `clicks_90d` | Total GSC clicks over 90-day feature window |
| `ctr_90d` | clicks / impressions × 100 |
| `avg_position_90d` | Mean search position — position=0 excluded, floored at 1.0 |
| `sessions_90d` | Total GA4 sessions — zero-filled when `has_ga4_data=0` |
| `pageviews_90d` | Total GA4 pageviews — zero-filled when `has_ga4_data=0` |
| `engaged_sessions_90d` | Total GA4 engaged sessions — zero-filled when `has_ga4_data=0` |
| `organic_sessions_90d` | Total organic sessions — zero-filled when `has_ga4_data=0` |
| `impressions_last30` | GSC impressions in March 2026 only — recency signal |
| `impressions_first60` | GSC impressions in Jan–Feb 2026 — baseline signal |
| `momentum_pct` | Trend direction over 90-day window — capped at 99th percentile (14,572.4%) |
| `active_days_90d` | Distinct days with GSC data in window — consistency signal |
| `has_ga4_data` | 1 if client has GA4 integration — structural missingness flag |
| `has_momentum` | 1 if `impressions_first60` > 0 — structural missingness flag |

### Context (identifiers — not features)
| Column | Use |
|---|---|
| `client_hash_id` | Grouped train/test split only |
| `content_hash_id` | Row identification only |

### Excluded
| Column | Reason |
|---|---|
| `impressions_mar` | Used to construct label — direct leakage |
| `impressions_apr` | Used to construct label — direct leakage |
| `pct_change` | Used to construct label — direct leakage |
| `trend_direction` | Derived from label — direct leakage |
| `trend_pct` | Derived from label — direct leakage |

In [2]:
feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]

print("=== FIELD VERIFICATION ===")
print(f"\nLabel column present:     {'is_declining_label' in dataset.columns}")
print(f"Feature columns present:  {all(c in dataset.columns for c in feature_cols)}")
print(f"Context columns present:  {all(c in dataset.columns for c in ['client_hash_id', 'content_hash_id'])}")

leak_cols = ['impressions_mar', 'impressions_apr', 'pct_change', 'trend_direction', 'trend_pct']
print(f"\nLeak columns absent:      {not any(c in dataset.columns for c in leak_cols)}")
print(f"\nAll columns in dataset:")
for col in dataset.columns:
    role = 'LABEL' if col == 'is_declining_label' \
        else 'CONTEXT' if col in ['client_hash_id', 'content_hash_id'] \
        else 'FEATURE'
    print(f"  {role:8s}  {col}")

=== FIELD VERIFICATION ===

Label column present:     True
Feature columns present:  True
Context columns present:  True

Leak columns absent:      True

All columns in dataset:
  CONTEXT   client_hash_id
  CONTEXT   content_hash_id
  LABEL     is_declining_label
  FEATURE   impressions_90d
  FEATURE   clicks_90d
  FEATURE   ctr_90d
  FEATURE   avg_position_90d
  FEATURE   sessions_90d
  FEATURE   pageviews_90d
  FEATURE   engaged_sessions_90d
  FEATURE   organic_sessions_90d
  FEATURE   impressions_last30
  FEATURE   impressions_first60
  FEATURE   momentum_pct
  FEATURE   active_days_90d
  FEATURE   has_ga4_data
  FEATURE   has_momentum


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
print("=== GRAIN CHECK ===")
print("Expected: one row per client_hash_id + content_hash_id pair")

duplicates = dataset.groupby(['client_hash_id', 'content_hash_id']).size()
max_count = duplicates.max()
print(f"Max rows per pair: {max_count}")
print(f"Grain is clean: {max_count == 1}")

print("=== COUNTS AND LABEL DISTRIBUTION ===")
print(f"Total items:      {len(dataset):,}")
print(f"Total clients:    {dataset['client_hash_id'].nunique()}")
print(f"Total content:    {dataset['content_hash_id'].nunique():,}")
print(f"\nDecl ining (1):  {dataset['is_declining_label'].sum():,} "
      f"({dataset['is_declining_label'].mean():.1%})")
print(f"Stable    (0):   {(dataset['is_declining_label']==0).sum():,} "
      f"({(1-dataset['is_declining_label'].mean()):.1%})")
print(f"\nNote: near-balanced classes — no resampling required.")
print("=== MISSING VALUES ===")
nulls = dataset[feature_cols].isnull().sum()
if nulls.sum() == 0:
    print("No null values in any feature column.")
else:
    print(nulls[nulls > 0])

print(f"\nGA4 zero-fill check:")
print(f"  Rows with has_ga4_data=0: {(dataset['has_ga4_data']==0).sum():,}")
print(f"  Of those, sessions_90d=0: "
      f"{((dataset['has_ga4_data']==0) & (dataset['sessions_90d']==0)).sum():,}")
print(f"  Alignment: "
      f"{((dataset['has_ga4_data']==0) & (dataset['sessions_90d']==0)).sum() == (dataset['has_ga4_data']==0).sum()}")

print("=== FEATURE RANGES ===")
print(dataset[feature_cols].describe().round(2).to_string())

=== GRAIN CHECK ===
Expected: one row per client_hash_id + content_hash_id pair
Max rows per pair: 1
Grain is clean: True
=== COUNTS AND LABEL DISTRIBUTION ===
Total items:      103,691
Total clients:    42


Total content:    103,691

Decl ining (1):  38,866 (37.5%)
Stable    (0):   64,825 (62.5%)

Note: near-balanced classes — no resampling required.
=== MISSING VALUES ===
No null values in any feature column.

GA4 zero-fill check:
  Rows with has_ga4_data=0: 25,826
  Of those, sessions_90d=0: 25,826
  Alignment: True
=== FEATURE RANGES ===


       impressions_90d  clicks_90d    ctr_90d  avg_position_90d  sessions_90d  pageviews_90d  engaged_sessions_90d  organic_sessions_90d  impressions_last30  impressions_first60  momentum_pct  active_days_90d  has_ga4_data  has_momentum
count        103691.00   103691.00  103691.00         103691.00     103691.00      103691.00             103691.00              103691.0           103691.00            103691.00     103691.00        103691.00     103691.00     103691.00
mean           5725.70       17.89       0.28             13.64         16.85          21.60                  0.57                  10.9             2681.25              3044.45        411.27            65.44          0.75          0.85
std           14998.54       80.09       0.44             13.99         67.32          95.09                  3.64                  62.5             6883.35              9157.98       1654.38            27.85          0.43          0.36
min              50.00        0.00       0.00       

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

- **Label is a proxy:** impression change measures search visibility, not confirmed
  editorial need. A page can lose impressions due to seasonality, algorithm updates,
  or competition — not only content quality. All outputs are directional and
  require human review.

- **Single time window:** one feature/label window pair. Results may not generalize
  across different time periods or seasonal patterns. A page that declines in
  April 2026 may behave differently in other months.

- **Client coverage:** 42 clients with varying history depth and GA4 availability.
  About a quarter of all rows (25,826 of 103,691) have no GA4 data — GA4 features
  carry no signal for those clients. The `has_ga4_data` flag allows the model to
  learn this boundary.

- **Minimum impressions filter:** 50 impressions per month. Content below this
  threshold was excluded. Low-impression content has insufficient search presence
  to generate actionable editorial recommendations, but the threshold also means
  the model only covers established content — not new pages.

- **Observed associations only:** this work does not establish causality between
  content signals and impression decline. No claim is made about Google's ranking
  algorithm or the causal impact of editorial refresh on recovery.

In [4]:
print("=== DATA CONTRACT FINAL VERIFICATION ===")
print(f"Dataset path:       work/outputs/dataset.csv")
print(f"Shape:              {dataset.shape}")
print(f"Items:              {len(dataset):,}")
print(f"Features:           {len(feature_cols)}")
print(f"Label:              is_declining_label")
print(f"Declining (1):      {dataset['is_declining_label'].sum():,} "
      f"({dataset['is_declining_label'].mean():.1%})")
print(f"Stable (0):         {(dataset['is_declining_label']==0).sum():,} "
      f"({(1-dataset['is_declining_label'].mean()):.1%})")
print(f"Null values:        {dataset[feature_cols].isnull().sum().sum()}")
print(f"Clients:            {dataset['client_hash_id'].nunique()}")
print(f"Grain clean:        {dataset.groupby(['client_hash_id','content_hash_id']).size().max() == 1}")
print(f"Leak columns:       {any(c in dataset.columns for c in leak_cols)}")
print(f"\nContract status:    READY FOR BASELINE (w04)")

=== DATA CONTRACT FINAL VERIFICATION ===
Dataset path:       work/outputs/dataset.csv
Shape:              (103691, 17)
Items:              103,691
Features:           14
Label:              is_declining_label
Declining (1):      38,866 (37.5%)
Stable (0):         64,825 (62.5%)
Null values:        0
Clients:            42


Grain clean:        True
Leak columns:       False

Contract status:    READY FOR BASELINE (w04)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.